In [12]:
from sumo_rl import SumoEnvironment

In [13]:
import numpy as np
from sumo_rl.environment.observations import ObservationFunction
import gymnasium as gym

class MyObservation(ObservationFunction):
    def __init__(self, traffic_signal):
        super().__init__(traffic_signal)
        self.traffic_signal = traffic_signal 

    def __call__(self):
        ts = self.traffic_signal
        sumo = ts.sumo

        incoming_lanes = ts.lanes
        outgoing_lanes = ts.out_lanes

        # ---- Pressure ----
        incoming = sum(sumo.lane.getLastStepVehicleNumber(l) for l in incoming_lanes)
        outgoing = sum(sumo.lane.getLastStepVehicleNumber(l) for l in outgoing_lanes)
        pressure = (incoming - outgoing) / 50.0

        # ---- Waiting Time ----
        waiting_time = sum(sumo.lane.getWaitingTime(l) for l in incoming_lanes) / 1000.0

        # ---- Halted Vehicles ----
        halted = sum(sumo.lane.getLastStepHaltingNumber(l) for l in incoming_lanes) / 50.0

        # ---- One-hot phase encoding ----
        n_phases = ts.num_green_phases  # FIX: Use num_green_phases instead of len(ts.phases)
        phase_idx = ts.green_phase      # FIX: Use green_phase instead of phase
        phase_onehot = np.zeros(n_phases, dtype=np.float32)
        phase_onehot[phase_idx] = 1.0

        # ---- Concatenate all features ----
        obs = np.concatenate([[pressure, waiting_time, halted], phase_onehot])
        return obs.astype(np.float32)

    def observation_space(self):
        ts = self.traffic_signal
        n_phases = ts.num_green_phases  # FIX: Use num_green_phases instead of len(ts.phases)
        
        # 3 scalars + n_phases one-hot
        dim = 3 + n_phases
        return gym.spaces.Box(low=0, high=1.0, shape=(dim,), dtype=np.float32)

In [14]:
env = SumoEnvironment(
        net_file="./env/3x3Grid2lanes.net.xml",
        route_file="./env/routes14000.rou.xml",
        observation_class=MyObservation,
        use_gui=True,
        num_seconds=100000,
    )

 Retrying in 1 seconds
Step #0.00 (0ms ?*RT. ?UPS, TraCI: 27ms, vehicles TOT 0 ACT 0 BUF 0)                     


In [15]:
env.reset()

 Retrying in 1 seconds


{'0': array([0., 0., 0., 1., 0., 0., 0.], dtype=float32),
 '1': array([0., 0., 0., 1., 0., 0., 0.], dtype=float32),
 '2': array([0., 0., 0., 1., 0., 0., 0.], dtype=float32),
 '3': array([0., 0., 0., 1., 0., 0., 0.], dtype=float32),
 '4': array([0., 0., 0., 1., 0., 0., 0.], dtype=float32),
 '5': array([0., 0., 0., 1., 0., 0., 0.], dtype=float32),
 '6': array([0., 0., 0., 1., 0., 0., 0.], dtype=float32),
 '7': array([0., 0., 0., 1., 0., 0., 0.], dtype=float32),
 '8': array([0., 0., 0., 1., 0., 0., 0.], dtype=float32)}

In [16]:
env.ts_ids

['0', '1', '2', '3', '4', '5', '6', '7', '8']

In [17]:
env.observation_spaces("0")

Box(0.0, 1.0, (7,), float32)

In [29]:
actions = {
    ts: env.action_space.sample()
    for ts in env.ts_ids
}

env.step(actions)

({'0': array([0.06 , 0.016, 0.02 , 0.   , 0.   , 0.   , 1.   ], dtype=float32),
  '1': array([0.02, 0.  , 0.  , 0.  , 0.  , 0.  , 1.  ], dtype=float32),
  '2': array([0.04, 0.01, 0.02, 0.  , 0.  , 0.  , 1.  ], dtype=float32),
  '3': array([0., 0., 0., 1., 0., 0., 0.], dtype=float32),
  '4': array([0., 0., 0., 0., 1., 0., 0.], dtype=float32),
  '5': array([0.06, 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ], dtype=float32),
  '6': array([0.04 , 0.001, 0.02 , 0.   , 0.   , 1.   , 0.   ], dtype=float32),
  '7': array([0.06, 0.  , 0.  , 0.  , 0.  , 0.  , 1.  ], dtype=float32),
  '8': array([0.12 , 0.081, 0.08 , 0.   , 0.   , 0.   , 1.   ], dtype=float32)},
 {'0': -0.05,
  '1': 0.0,
  '2': -0.05,
  '3': 0.06,
  '4': 0.0,
  '5': 0.0,
  '6': 0.019999999999999997,
  '7': 0.0,
  '8': -0.20000000000000007},
 {'0': False,
  '1': False,
  '2': False,
  '3': False,
  '4': False,
  '5': False,
  '6': False,
  '7': False,
  '8': False,
  '__all__': False},
 {'step': 50.0,
  'system_total_stopped': 7,
  'system

In [31]:
env.traffic_signals['0'].lanes

['0Ni_0', '0Ni_1', '0Ei_0', '0Ei_1', '0Si_0', '0Si_1', '0Wi_0', '0Wi_1']

In [32]:
env.close()